# 실습 1. Python 텍스트 데이터 처리 (Day 1 / 90분)

> 시나리오: **쇼핑몰 리뷰 데이터 정제** — `data/reviews.csv`
>
> Day 1(Pandas·정규식)은 이 저장소의 **로컬 노트북**으로 진행한다.

## Task
1. CSV 로드 후 **결측·이상치 리포트** 작성
2. `text` 컬럼 정제 — URL/특수문자/공백 정규화
3. **정규식**으로 광고 패턴(`[광고]`, `^협찬`, `구매처:`) 제거
4. `rating` 별 평균 텍스트 길이 / 단어 수 집계
5. 정제 결과를 `reviews_clean.parquet` 으로 저장

In [4]:
import re
# 이메일 추출
text = "asda@gmail.com  dsfafdfdfsfdsfadf, affdasfdsfsdafdsf rfgregrgre@afefafefe.com"
emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
print(emails)
hello = re.findall(r"asd",text)
print(hello)
# 전화번호 정규화 (010-1234-5678 / 01012345678 / 010 1234 5678)
phone = re.sub(r"[\s-]", "", "010 1234-5678") # "01012345678"
# 한글 추출
korean = re.findall(r"[가-힣]+", "Hello 안녕 World 세상")
# ['안녕', '세상']
# 명명된 그룹 — 가독성 ↑
m = re.match(r"(?P<year>\d{4})-(?P<month>\d{2})", "2026-05")
m.group("year") # "2026"

['asda@gmail.com', 'rfgregrgre@afefafefe.com']
['asd']


'2026'

In [6]:
import pandas as pd

# data/reviews.csv 가 없으면: 터미널에서 `python data/make_reviews.py`
df = pd.read_csv("../data/reviews.csv")
print(df.shape)
df.head()
#print(df.head())

(10000, 5)
   id     user                                          text  rating  \
0   1  user087            협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍     2.0   
1   2  user072         협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍     4.0   
2   3  user021                         배송이 일주일이나 걸렸습니다 별로 ㅠㅠ     2.0   
3   4  user045  [광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡     4.0   
4   5  user047    [광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ     5.0   

            created_at  
0     2026/06/18 02:37  
1           23.02.2026  
2  2026-06-14T10:17:00  
3                  NaN  
4           2026-05-07  


## 1) 결측·이상치 리포트

In [12]:
df.info()
print("\n결측치:\n", df.isna().sum())
print("\nrating 분포:\n", df["rating"].value_counts(dropna=False))

# created_at 원본을 보존하고 다양한 형식으로 재파싱(dayfirst=True 포함)
df["created_at_raw"] = df["created_at"].astype(str).str.strip().replace("nan", "")
df["created_at"] = pd.to_datetime(df["created_at_raw"], errors="coerce", dayfirst=True)
print("\n날짜 파싱 실패(NaT):", df["created_at"].isna().sum(), "건")
# 파싱 실패(=NaT)인 행은 삭제
df = df.dropna(subset=["created_at"])

<class 'pandas.DataFrame'>
Index: 8000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              8000 non-null   int64         
 1   user            8000 non-null   str           
 2   text            8000 non-null   str           
 3   rating          7596 non-null   float64       
 4   created_at      8000 non-null   datetime64[us]
 5   created_at_raw  8000 non-null   str           
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 1.2 MB

결측치:
 id                  0
user                0
text                0
rating            404
created_at          0
created_at_raw      0
dtype: int64

rating 분포:
 rating
4.0    1903
5.0    1850
1.0    1379
2.0    1328
3.0    1136
NaN     404
Name: count, dtype: int64

날짜 파싱 실패(NaT): 0 건


/tmp/ipykernel_30878/2556134276.py:7: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["created_at"] = pd.to_datetime(df["created_at_raw"], errors="coerce", dayfirst=True)


## 2) 텍스트 정제 파이프라인 (슬라이드 `clean()` 참고)

`.str` 로 컬럼 전체를 한 번에 정제한다. 정규식 패턴 풀이:
- `https?://\S+` → http/https 로 시작하는 URL (공백 전까지)
- `[^\w가-힣\s]` → 단어문자(`\w`)·한글(`가-힣`)·공백(`\s`) **이 아닌** 것 = 특수문자/이모지
- `\s+` → 연속 공백을 1칸으로

In [14]:
import re

def clean(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .str.strip()
         .str.replace(r"https?://\S+", "", regex=True)        # URL 제거
         .str.replace(r"[^\w가-힣\s]", " ", regex=True)        # 특수문자 제거 (가-힣 = U+AC00~U+D7A3)
         .str.replace(r"\s+", " ", regex=True)                 # 공백 정규화
         .str.strip()
    )

df["clean_text"] = clean(df["text"])
df[["text", "clean_text"]].head()

,text,clean_text
0,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍,협찬 받아 작성한 후기입니다 생각보다 너무 작고 부실해요
1,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,협찬 받아 작성한 후기입니다 배송이 정말 빨라서 깜짝 놀랐어요
2,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,배송이 일주일이나 걸렸습니다 별로
4,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,광고 지금 구매하면 사은품 증정 배송이 정말 빨라서 깜짝 놀랐어요
5,[광고] 지금 구매하면 사은품 증정! 가성비 최고입니다 또 살게요 👍,광고 지금 구매하면 사은품 증정 가성비 최고입니다 또 살게요


## 3) 광고 패턴 제거 (정규식)

In [16]:
AD = re.compile(r"\[광고\]|^협찬|구매처\s*:")
is_ad = df["text"].str.contains(AD)
print(f"광고성 리뷰: {is_ad.sum()}건")
# TODO: 광고 문구만 제거할지, 행 자체를 버릴지 결정하고 처리
df[is_ad][["text", "clean_text"]].head()
df = df[~is_ad].copy()
print(f"광고성 리뷰 제거 후 건수: {len(df)}건")

광고성 리뷰: 3942건
광고성 리뷰 제거 후 건수: 4058건


## 4) rating 별 텍스트 길이 / 단어 수 집계

In [17]:
df["length"] = df["clean_text"].str.len()
df["words"] = df["clean_text"].str.split().str.len()
df.groupby("rating")[["length", "words"]].mean()

,length,words
rating,,
1.0,18.173977,4.837719
2.0,18.183644,4.819225
3.0,14.261986,3.945205
4.0,17.815401,4.497890
5.0,17.891693,4.503680


## 5) 저장 — 실습 2의 입력이 된다

In [18]:
df.to_parquet("../data/reviews_clean.parquet", index=False)
print("saved: data/reviews_clean.parquet")

saved: data/reviews_clean.parquet
